# 1.  Building Block 1 - ZCRMS Normalizer

## ZCRMSNorm (Zero-Centered RMSNorm)

**ZCRMSNorm** is the normalizer used in **Cactus Needle**. 

### Step-by-Step Process
Given an input vector $x$ (where the last dimension is the one being normalized, e.g., `d_model`):

#### 1. Zero-center
$$x \leftarrow x - \text{mean}(x)$$
Subtract the mean so the vector has a mean of zero.

#### 2. RMS Normalize
$$\text{rms} = \sqrt{\text{mean}(x^2) + \epsilon}$$
$$x \leftarrow \frac{x}{\text{rms}}$$
Divide by the root-mean-square so the vector has unit RMS.

#### 3. Learned Scale
$$\text{output} = x \times (1 + \text{scale})$$
* `scale` is a learned parameter that **starts at zero**.
* At initialization, the multiplier is exactly `1.0`, so the block begins as a pure normalizer. 
* During training, it can stretch or shrink individual dimensions.

---

### Why "Zero-Centered" + RMS?

* **Ordinary RMSNorm** only does Step 2 (no mean subtraction).
* **LayerNorm** does mean subtraction + variance normalization + affine transform.
* **ZCRMSNorm** combines the mean subtraction of LayerNorm with the RMS scaling of RMSNorm, and uses a zero-initialized scale so the residual stream is not disturbed at the start of training.

### Architectural Context
In the **Cactus architecture**, it sits at the beginning of every attention block (before the at

In [1]:
# Cell 1 — imports
import torch
from needle import ZCRMSNorm

In [2]:
# Cell 2 — test ZCRMSNorm
norm = ZCRMSNorm(dim=16)
x = torch.randn(2, 8, 16)
y = norm(x)
print("shape:", y.shape)
print("scale init max:", norm.scale.abs().max().item())
assert y.shape == x.shape
print("ZCRMSNorm OK")

shape: torch.Size([2, 8, 16])
scale init max: 0.0
ZCRMSNorm OK


# 2.  Building Block 2 - RoPE

## RoPE (Rotary Position Embedding)

**RoPE** encodes position by rotating pairs of features in the query and key vectors, instead of adding a separate position vector.

### How it works (intuition)
* Each pair of dimensions in a head is treated as a 2-D plane.
* For position $t$, we rotate that plane by an angle proportional to $t$.
* Different frequency bands rotate at different speeds (controlled by the $\theta$ base).
* When we compute the attention score $q \cdot k$, the relative angle between them automatically appears in the dot product.
* Absolute positions are never stored; only relative relationships matter.

### How it differs from common alternatives

| Method | How position is injected | Relative position? | Extrapolation? | Extra parameters? |
| :--- | :--- | :--- | :--- | :--- |
| **Learned absolute** | Add a learned vector per position | No (absolute only) | Poor beyond training length | Yes (one vector per position) |
| **Sinusoidal absolute** | Add fixed sin/cos vector | Weakly | Limited | No |
| **RoPE** | Rotate Q and K features | Yes (built into dot product) | Better | No |
| **Relative bias (e.g., T5)** | Add a bias to attention scores | Yes | Decent | Yes (bias table) |

### Why it is useful / often better

* **Relative position is free:** The attention score between two tokens depends on their distance, not on their absolute indices. This matches how language (and often images) works.
* **No extra parameters:** Unlike learned embeddings or relative bias tables, RoPE adds zero trainable weights.
* **Better length extrapolation:** Models trained with RoPE often degrade more gracefully when you give them sequences longer than those seen in training (important for transformers).
* **Works well with modern attention:** It is the default in LLaMA, Mistral, Gemma, and also in **Cactus Needle**. Needle uses it on self-attention (and turns it off for cross-attention, because source and target have different position tracks).

### In our Cactus code

* We apply RoPE only to queries and keys of **self-attention**.
* **Cross-attention** does not use RoPE (encoder and decoder positions are independent).
* The three helpers (`rotate_half`, `rope_cos_sin`, `apply_rope`) are exactly the implementation of this rotation.
* That is why we put RoPE right after the normalizer in the block order: every self-attention layer needs it.

In [3]:
from needle import rope_cos_sin, apply_rope

cos, sin = rope_cos_sin(seq_len=8, head_dim=16)
x = torch.randn(2, 4, 8, 16)          # (batch, heads, time, dim)
y = apply_rope(x, cos, sin)
print("shape:", y.shape)
assert y.shape == x.shape
print("RoPE OK # RoPE does not change the shape, it changes the values of the input tensor")

shape: torch.Size([2, 4, 8, 16])
RoPE OK # RoPE does not change the shape, it changes the values of the input tensor


# 3.  Building Block 3 - scaled_dot_product_attention

## Scaled Dot-Product Attention — in general

This is the core operation inside almost every modern transformer (Vaswani et al., 2017). 

Given three tensors derived from the input:
* **Query $Q$** — "what am I looking for?"
* **Key $K$** — "what do I contain?"
* **Value $V$** — "what information do I actually pass on?"

The attention is computed as:
$$
\text{Attention}(Q, K, V) = \text{softmax}\left( \frac{Q K^\top}{\sqrt{d_k}} \right) V
$$

### Step by step:
1. **Dot product $Q K^\top$**  
   Measures similarity between every query and every key.  
   *Shape: `(batch, heads, q_len, k_len)`*
2. **Scale by $\sqrt{d_k}$**  
   Prevents the dot products from becoming too large (which would push the softmax into regions with tiny gradients).
3. **Mask (optional)**  
   Sets forbidden positions (padding or future tokens) to $-\infty$ so the softmax gives them zero weight.
4. **Softmax**  
   Turns the scores into a probability distribution over the keys (weights sum to 1).
5. **Weighted sum of the values**  
   Each query receives a mixture of the value vectors, weighted by how relevant the corresponding keys were.

**Result:** A tensor of the same shape as the queries, where every position has gathered information from the positions it attended to.

---

### How it is used in the Cactus / Needle architecture

Cactus Needle is a pure-attention encoder–decoder. Almost every block’s main computation is this scaled dot-product attention:

| Place | What Q, K, V come from | Mask | RoPE? |
| :--- | :--- | :--- | :--- |
| **Encoder self-attention** | All from the same encoder hidden state | Optional padding | Yes |
| **Decoder self-attention** | All from the decoder hidden state | Causal (+ padding) | Yes |
| **Decoder cross-attention** | Q from decoder, K/V from encoder memory | Optional padding on source | No |

### In our code

* The function `scaled_dot_product_attention` is exactly the formula above.
* The module `GroupedQueryAttention` (next block) is the wrapper that:
  1. Projects the input into $Q, K, V$,
  2. Applies RoPE when needed,
  3. Calls this attention function,
  4. Projects the result back to `d_model`.

Because Needle has **no FFN**, this attention operation (plus the norm and the gated residual) is essentially the *entire* content of each layer. That is why we implement it carefully and test it on its own.

In [4]:
from needle import scaled_dot_product_attention

B, H, T, D = 2, 4, 8, 16
q = torch.randn(B, H, T, D)
k = torch.randn(B, H, T, D)
v = torch.randn(B, H, T, D)

out = scaled_dot_product_attention(q, k, v)
print("shape:", out.shape)
assert out.shape == (B, H, T, D)
print("attention OK")

shape: torch.Size([2, 4, 8, 16])
attention OK


# 4.  Building Block 4 - Grouped-Query Attention Module

## Grouped-Query Attention (GQA) Module

This is the module that wraps the attention function we just wrote. It is the actual layer used inside every Cactus block.

### What it does
* Projects the input into Query, Key, and Value with linear layers.
* Uses **Grouped-Query Attention (GQA)**: fewer key/value heads than query heads (saves parameters and memory).
* Applies **RoPE** to $Q$ and $K$ when it is self-attention (`use_rope=True`).
* Calls `scaled_dot_product_attention`.
* Projects the result back to `d_model`.

---

### Self-Attention vs. Cross-Attention

This single module is designed to handle both types of attention seamlessly, depending on the inputs provided:

#### 1. Self-Attention
* **How it works:** Every token in a sequence looks at every other token in the *same* sequence. The Query, Key, and Value all come from the exact same input tensor.
* **Purpose:** It allows the model to build context and relationships within a single text (e.g., resolving pronouns like "it" or "they", understanding syntax, or linking related concepts).
* **Where it's used:** In both the encoder and the decoder.

#### 2. Cross-Attention (Encoder-Decoder Attention)
* **How it works:** Tokens in the *decoder* sequence look at the tokens in the *encoder* sequence. The Query comes from the decoder, while the Keys and Values come from the encoder's memory.
* **Purpose:** It allows the decoder to "read" and use the information encoded from the source text to generate the target text (e.g., translating a sentence by looking at the corresponding words in the original language).
* **Where it's used:** Only in the decoder (in encoder-decoder architectures like Cactus Needle).

By simply changing the inputs passed to this module, we can switch between letting a sequence understand itself (**self-attention**) and letting the decoder read the encoder's output (**cross-attention**).

In [5]:
from needle import GroupedQueryAttention

attn = GroupedQueryAttention(d_model=64, n_heads=8, n_kv_heads=2, use_rope=True)
x = torch.randn(2, 16, 64)          # (batch, seq, d_model)
y = attn(x, x)                      # self-attention
print("shape:", y.shape)
assert y.shape == x.shape
print("GroupedQueryAttention OK")

shape: torch.Size([2, 16, 64])
GroupedQueryAttention OK


## Gated Residual

Cactus Needle does not use a plain residual connection ($x + \text{update}$). Instead, it uses a **gated residual**:

$$
x \leftarrow x + \sigma(g) \cdot \text{update}
$$

where $g$ is a learned scalar (or vector) that starts at $0$.
* At initialization, $\sigma(0) = 0.5$.
* Because $g$ begins at zero, the gate has the flexibility to learn to open or close during training.
* In practice, Needle initializes it so the residual starts nearly closed and gradually opens as training progresses.

### Why Needle uses it

* **Stability:** Keeps the residual stream stable at the beginning of training, preventing early divergence or gradient explosion.
* **Adaptive Mixing:** Lets the model learn exactly how much of each sub-layer (e.g., attention) to mix into the residual stream.
* **"Start Quiet" Philosophy:** Together with the zero-initialized scale inside **ZCRMSNorm**, it is a core part of the *“start quiet, then open up”* design philosophy of the Cactus architecture.

In [6]:
from needle import GatedResidual

gate = GatedResidual(dim=16)
x = torch.randn(2, 8, 16)
update = torch.randn(2, 8, 16)
y = gate(x, update)

print("shape:", y.shape)
assert y.shape == x.shape
print("gate init (sigmoid):", torch.sigmoid(gate.gate).mean().item())  # ~0.5
print("GatedResidual OK")

shape: torch.Size([2, 8, 16])
gate init (sigmoid): 0.5
GatedResidual OK


# Encoder - Decoder Block


These are the basic repeating layers of the Cactus encoder–decoder architecture. Because we are building the **pure version**, each block contains *only* attention mechanisms (no feed-forward network).

---

### EncoderBlock
* **Input:** A sequence of patch embeddings (or token embeddings).
* **Output:** A richer sequence of the same length and dimension.

**Inside one block:**
1. **ZCRMSNorm** — Normalizes the residual stream.
2. **Self-attention (with RoPE)** — Every position looks at every other position in the same sequence and gathers contextual information.
3. **Gated residual** — Adds the attention output back to the original stream, controlled by a learned gate.

---

### DecoderBlock
* **Input:** The current decoder state (what we are reconstructing so far) and the encoder memory.
* **Output:** An updated decoder state.

**Inside one block, there are two distinct attention steps:**

1. **Causal Self-Attention (with RoPE)**  
   * Each position can only look at previous positions (or itself).  
   * This keeps the reconstruction autoregressive (if desired) or simply allows the decoder tokens to communicate with each other.  
   * *Followed by a gated residual.*

2. **Cross-Attention (no RoPE)**  
   * **Queries** come from the decoder.  
   * **Keys and Values** come from the encoder memory.  
   * This is how the decoder actively "reads" the encoded face.  
   * *Followed by another gated residual.*

---

### How They Work Together (Image Autoencoder)

```text
Image Patches
     ↓
[EncoderBlock] × N   →   Memory (Encoded Face)
     ↓
[DecoderBlock] × M   →   Reconstructed Patch Features
     ↓
[Linear Head]        →   Pixels
```
---

The encoder compresses / contextualizes the input patches.

The decoder uses cross-attention to look at that memory and produces features that can be turned back into an image.

Because there is no FFN, almost all of the computation is these attention operations plus the norms and gates.

In [7]:
from needle import EncoderBlock, DecoderBlock
import torch

# EncoderBlock
enc = EncoderBlock(d_model=64, n_heads=8, n_kv_heads=2)
x = torch.randn(2, 16, 64)
y = enc(x)
print("encoder out:", y.shape)
assert y.shape == x.shape

# DecoderBlock
dec = DecoderBlock(d_model=64, n_heads=8, n_kv_heads=2)
tgt = torch.randn(2, 10, 64)
mem = torch.randn(2, 16, 64)
z = dec(tgt, mem)
print("decoder out:", z.shape)
assert z.shape == tgt.shape
print("Blocks OK")

encoder out: torch.Size([2, 16, 64])
decoder out: torch.Size([2, 10, 64])
Blocks OK


# Final Needle Back Bone
The individual layers are now are now stacked to create a full cactus needle architecture.


In [8]:
from needle import NeedleBackbone
import torch

backbone = NeedleBackbone(
    d_model=64,
    n_heads=8,
    n_kv_heads=2,
    n_encoder_layers=2,
    n_decoder_layers=2,
)
src = torch.randn(2, 16, 64)   # e.g. 16 patches
tgt = torch.randn(2, 16, 64)
out = backbone(src, tgt)
print("backbone out:", out.shape)
assert out.shape == tgt.shape
print("NeedleBackbone OK")

backbone out: torch.Size([2, 16, 64])
NeedleBackbone OK


In [9]:
from model import CactusImageAE
import torch

model = CactusImageAE(img_size=64, patch_size=8, d_model=64, n_encoder_layers=2, n_decoder_layers=2)
x = torch.rand(2, 3, 64, 64)
y = model(x)
print("output shape:", y.shape)
assert y.shape == x.shape
print("CactusImageAE OK")
print("params (M):", round(sum(p.numel() for p in model.parameters()) / 1e6, 3))

output shape: torch.Size([2, 3, 64, 64])
CactusImageAE OK
params (M): 0.087
